In [2]:
from typing import Annotated
from langchain_deepseek import ChatDeepSeek
from langchain_core.messages import AnyMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages 
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage

In [3]:
load_dotenv()

True

In [4]:
llm = ChatDeepSeek(model='deepseek-chat')

In [5]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [6]:
def chatnode(state: ChatState):
    decision = interrupt (
        {
            "type" : "approval",
            "reason" : "Model is about to view this question",
            "question" : state["messages"][-1].content,
            "instruction" : "Approve this question ? yes/no"
        }
    )
    if decision["approved"] == "no":
        return {"messages": [AIMessage(content="Not approved.")]}
    else:
        response = llm.invoke(state["messages"])
        return {"messages": [response]}

In [8]:
builder = StateGraph(ChatState)

builder.add_node("chat", chatnode)

builder.add_edge(START, "chat")
builder.add_edge("chat", END)

# Checkpointer is required for interrupts
checkpointer = MemorySaver()

# Compile the app
app = builder.compile(checkpointer=checkpointer)

In [9]:
# Create a new thread id for this conversation
config = {"configurable": {"thread_id": '1234'}}

# ---- STEP 1: user asks a question ----
initial_input = {
    "messages": [
        ("user", "Explain gradient descent in very simple terms.")
    ]
}

# Invoke the graph for the first time
result = app.invoke(initial_input, config=config)

In [10]:
result

{'messages': [HumanMessage(content='Explain gradient descent in very simple terms.', additional_kwargs={}, response_metadata={}, id='3fc1579b-0170-425b-83c8-2296f5c9436a')],
 '__interrupt__': [Interrupt(value={'type': 'approval', 'reason': 'Model is about to view this question', 'question': 'Explain gradient descent in very simple terms.', 'instruction': 'Approve this question ? yes/no'}, id='b1638444727cf42c6c84ebec2dc27aa1')]}

In [11]:
message = result['__interrupt__'][0].value
message

{'type': 'approval',
 'reason': 'Model is about to view this question',
 'question': 'Explain gradient descent in very simple terms.',
 'instruction': 'Approve this question ? yes/no'}

In [12]:
user_input = input(f"\nBackend message - {message} \n Approve this question? (y/n): ")


Backend message - {'type': 'approval', 'reason': 'Model is about to view this question', 'question': 'Explain gradient descent in very simple terms.', 'instruction': 'Approve this question ? yes/no'} 
 Approve this question? (y/n):  yes


In [13]:
#second pass
final_result = app.invoke(
    Command(resume={"approved":user_input}),
    config=config
)

In [14]:
print(final_result["messages"][-1].content)

Imagine you're standing on a mountain in thick fog. You can only see your feet, not the whole mountain. Your goal is to get to the bottom of the valley (the lowest point). 

Since you can't see the whole landscape, what do you do? **You feel the ground with your foot and take a step in the direction that goes slightly downhill.** Then you do it again. And again. 

Gradient descent is exactly that, but for math and computers.

Here’s the step-by-step breakdown:

1. **You have a mistake (the "loss"):** 
   Imagine you are trying to guess a number. Your guess is wrong. The amount you are wrong is your "error" or "loss." In the mountain analogy, your *height* is your error. High up = lots of error. Low down = no error.

2. **You check the slope (the "gradient"):**
   You look at your current spot and ask, "If I change my guess a tiny bit, will my error go up or go down?" The direction that makes the error go down is the "gradient." It’s like tapping your foot on the mountain to feel which 